In [ ]:
# Cell 1
# Mục đích: tạo kênh kết nối BigQuery và 2 hàm an toàn (check = dry run, stats = chạy thật có trần byte).
from google.cloud import bigquery

PROJECT = "bq-learning-510104"
client = bigquery.Client(project=PROJECT, location="US")  # location phải khớp vùng dataset (US)

def mb(n):
    # Đổi byte sang MB; trả None nếu n là None để lỗi "đọc byte quá sớm" lộ ra
    return None if n is None else round(n / 1024 / 1024, 2)

def check(sql):
    # Dry run: chỉ ước lượng, không đọc dữ liệu, không tốn tiền
    cfg = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
    job = client.query(sql, job_config=cfg)
    print(f"Se quet: {mb(job.total_bytes_processed)} MB")

def stats(sql, max_mb=100):
    # Chạy thật; maximum_bytes_billed là cầu chì (vượt thì BigQuery từ chối, không tính tiền)
    cfg = bigquery.QueryJobConfig(use_query_cache=False,
                                  maximum_bytes_billed=max_mb * 1024 * 1024)
    job = client.query(sql, job_config=cfg)
    df = job.result().to_dataframe()   # result() chờ job xong rồi mới đọc byte
    print(f"processed: {mb(job.total_bytes_processed)} MB | billed: {mb(job.total_bytes_billed)} MB")
    return df

In [ ]:
# Cell 2
# Mục đích: tính bằng SQL đúng 5 chỉ số của 5 measure, trên cùng bảng mart mà Power BI đã Import,
# để đối chiếu từng số.
sql = """
SELECT
  COUNT(*)                                             AS order_items,   -- COUNTROWS
  COUNT(DISTINCT order_id)                             AS orders,        -- DISTINCTCOUNT
  ROUND(SUM(sale_price), 2)                            AS net_revenue,   -- SUM
  ROUND(SUM(gross_profit), 2)                          AS gross_profit,  -- SUM
  ROUND(SAFE_DIVIDE(SUM(gross_profit), SUM(sale_price)), 4) AS margin    -- DIVIDE: tổng chia tổng
FROM `bq-learning-510104.dbt_dev.fct_net_sales`
"""
check(sql)   # xem số MB trước, nhỏ thì mới chạy Cell 3

Se quet: 1.04 MB


In [ ]:
# Cell 3
# Mục đích: chạy thật và in kết quả. In len(df) trước vì bảng rỗng thì mọi thứ phía sau im lặng.
df = stats(sql)
print(len(df))   # mong đợi 1 dòng
df

process: 1.04 MB | billed: 10.0 MB
1


,order_items,orders,net_revenue,gross_profit,margin
0,45529,31335,2685271.28,1391199.0,0.5181


In [6]:
# Cell 4
# Mục đích: đối chiếu bảng theo department (Phần 3, bước 3). Cell này đọc 2 bảng nên billed khoảng 20 MB.
sql_dep = """
SELECT
  p.department,
  COUNT(*)                   AS order_items,
  ROUND(SUM(f.sale_price), 2) AS net_revenue
FROM `bq-learning-510104.dbt_dev.fct_net_sales` AS f
LEFT JOIN `bq-learning-510104.dbt_dev.dim_products` AS p   -- LEFT JOIN để dòng không khớp không biến mất âm thầm
  ON f.product_id = p.product_id
GROUP BY p.department
ORDER BY p.department
"""
check(sql_dep)
df_dep = stats(sql_dep)
print(len(df_dep))   # mong đợi 2 dòng (Men, Women); nếu ra 3 thì có dòng department NULL
df_dep

Se quet: 1.09 MB
process: 1.09 MB | billed: 20.0 MB
2


,department,order_items,net_revenue
0,Men,23067,1427482.26
1,Women,22462,1257789.02
